## import all correlation data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
import networkx as nx

abs_segment_base = Path('correlation_matrices_segment_wise_absolute')
matrix_files = list(abs_segment_base.rglob('*.csv'))

if matrix_files:
    sample = pd.read_csv(matrix_files[0], index_col=0)
    print(f"Found {len(matrix_files)} absolute matrices. Sample shape: {sample.shape}")


Found 2632 absolute matrices. Sample shape: (19, 19)


## filter out our desired subjects with 12 segments

In [ ]:
# Filter subjects
subject_ids = [64, 65, 69, 70, 75, 77, 78, 79, 80, 81, 82, 83, 84, 86, 87, 89, 90, 94, 96, 103]
formatted_subjects = [f"sub-NORB{sid:05d}" for sid in subject_ids]

# Extract first 12 segments for session 1
n_segments = 12
subject_matrices = {}

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    if not subj_dir.exists():
        print(f"Warning: directory not found for {subj}")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    if len(csv_files) < n_segments:
        print(f"Warning: {subj} has less than {n_segments} segments")
        
    matrices = []
    for csv_file in csv_files[:n_segments]:
        df = pd.read_csv(csv_file, index_col=0)
        matrices.append(df.values)
        
    if matrices:
        avg_matrix = np.mean(matrices, axis=0)
        subject_matrices[subj] = pd.DataFrame(avg_matrix, index=df.index, columns=df.columns)
        
print(f"Processed {len(subject_matrices)} subjects")


Processed 20 subjects


## plot showing change in correlation for each edge (pair of nodes)

one graph for one subject session 1

- on the x axis, we have all the edges.
- for a single subject, there's 12 segments, so there will be 12 blocks on the y axis showing, segment 1 on the bottom, segment 12 on the top. 'hot' colourmap to show the value of correlation value of that edge (two nodes).

In [4]:
edge_labels = [
    'Fp1-Fp2', 'Fp1-F3', 'Fp1-F4', 'Fp1-F7', 'Fp1-F8', 'Fp1-FZ', 'Fp2-F3',
    'Fp2-F4', 'Fp2-F7', 'Fp2-F8', 'Fp2-FZ', 'F3-F4', 'F3-F7', 'F3-F8', 'F3-FZ',
    'F4-F7', 'F4-F8', 'F4-FZ', 'F7-F8', 'F7-FZ', 'F8-FZ', 'Fp1-C3', 'Fp1-C4',
    'Fp1-CZ', 'Fp2-C3', 'Fp2-C4', 'Fp2-CZ', 'F3-C3', 'F3-C4', 'F3-CZ', 'F4-C3',
    'F4-C4', 'F4-CZ', 'F7-C3', 'F7-C4', 'F7-CZ', 'F8-C3', 'F8-C4', 'F8-CZ',
    'FZ-C3', 'FZ-C4', 'FZ-CZ', 'Fp1-T3', 'Fp1-T4', 'Fp1-T5', 'Fp1-T6',
    'Fp2-T3', 'Fp2-T4', 'Fp2-T5', 'Fp2-T6', 'F3-T3', 'F3-T4', 'F3-T5', 'F3-T6',
    'F4-T3', 'F4-T4', 'F4-T5', 'F4-T6', 'F7-T3', 'F7-T4', 'F7-T5', 'F7-T6',
    'F8-T3', 'F8-T4', 'F8-T5', 'F8-T6', 'FZ-T3', 'FZ-T4', 'FZ-T5', 'FZ-T6',
    'Fp1-P3', 'Fp1-P4', 'Fp1-PZ', 'Fp2-P3', 'Fp2-P4', 'Fp2-PZ', 'F3-P3',
    'F3-P4', 'F3-PZ', 'F4-P3', 'F4-P4', 'F4-PZ', 'F7-P3', 'F7-P4', 'F7-PZ',
    'F8-P3', 'F8-P4', 'F8-PZ', 'FZ-P3', 'FZ-P4', 'FZ-PZ', 'Fp1-O1', 'Fp1-O2',
    'Fp2-O1', 'Fp2-O2', 'F3-O1', 'F3-O2', 'F4-O1', 'F4-O2', 'F7-O1', 'F7-O2',
    'F8-O1', 'F8-O2', 'FZ-O1', 'FZ-O2', 'C3-C4', 'C3-CZ', 'C4-CZ', 'C3-T3',
    'C3-T4', 'C3-T5', 'C3-T6', 'C4-T3', 'C4-T4', 'C4-T5', 'C4-T6', 'CZ-T3',
    'CZ-T4', 'CZ-T5', 'CZ-T6', 'C3-P3', 'C3-P4', 'C3-PZ', 'C4-P3', 'C4-P4',
    'C4-PZ', 'CZ-P3', 'CZ-P4', 'CZ-PZ', 'C3-O1', 'C3-O2', 'C4-O1', 'C4-O2',
    'CZ-O1', 'CZ-O2', 'T3-T4', 'T3-T5', 'T3-T6', 'T4-T5', 'T4-T6', 'T5-T6',
    'T3-P3', 'T3-P4', 'T3-PZ', 'T4-P3', 'T4-P4', 'T4-PZ', 'T5-P3', 'T5-P4',
    'T5-PZ', 'T6-P3', 'T6-P4', 'T6-PZ', 'T3-O1', 'T3-O2', 'T4-O1', 'T4-O2',
    'T5-O1', 'T5-O2', 'T6-O1', 'T6-O2', 'P3-P4', 'P3-PZ', 'P4-PZ', 'P3-O1',
    'P3-O2', 'P4-O1', 'P4-O2', 'PZ-O1', 'PZ-O2', 'O1-O2',
]

In [ ]:
# Create a directory to save the plots
output_dir = Path("changePlots/correlation")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the edge correlations for each segment
    # Shape: (12 segments, number of edges)
    heatmap_data = np.zeros((12, len(edge_labels)))

    # Populate the matrix with edge values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        for j, edge in enumerate(edge_labels):
            n1, n2 = edge.split('-')
            # Look up the correlation for the edge's nodes
            heatmap_data[i, j] = df.loc[n1, n2]

    # Segment 1 should be at the bottom, so we reverse the row order
    # (Seaborn plots row 0 at the top by default)
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(30, 8))
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=edge_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Pearson Correlation'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic
    # ---------------------------------------------------------
    # Combine X in data coordinates (columns) with Y in axes coordinates (plot box height)
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(edge_labels) + 1):
        # Extract the region initials (e.g., 'Fp1-F3' -> 'F-F')
        r_prev = f"{edge_labels[i-1].split('-')[0][0]}-{edge_labels[i-1].split('-')[1][0]}"
        r_curr = f"{edge_labels[i].split('-')[0][0]}-{edge_labels[i].split('-')[1][0]}" if i < len(edge_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(edge_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot (y=1.01 in axes coordinates)
            ax.text(mid_x, 1.01, r_prev, transform=trans, ha='center', va='bottom', fontsize=10, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    # Title with padding to make room for region labels
    plt.title(f"Segment-wise Correlation Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Edges")
    plt.ylabel("Segments")

    # Rotate x-axis labels to fit all 171 edges legibly
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all heatmaps with region labels!")


Saved: changePlots/correlation/sub-NORB00064_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00065_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00069_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00070_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00075_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00077_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00078_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00079_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00080_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00081_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00082_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00083_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00084_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00086_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00087_ses-1_heatmap.png
Saved: changePlots/correlation/sub-NORB00089_ses-1_heat

## plot showing change in node betweenness for each node

In [ ]:
# Create a directory to save the node betweenness plots
output_dir = Path("changePlots/node_betweenness")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

# Group nodes by region (F, C, T, P, O) to allow smooth cyan boundaries
node_labels = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'FZ', 
               'C3', 'C4', 'CZ', 'T3', 'T4', 'T5', 'T6', 
               'P3', 'P4', 'PZ', 'O1', 'O2']

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the node betweenness for each segment
    # Shape: (12 segments, 19 nodes)
    heatmap_data = np.zeros((12, len(node_labels)))

    # Populate the matrix with node betweenness values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
        
        # Calculate distance: 1 / correlation
        for u, v, d in G.edges(data=True):
            weight = d.get('weight', 0)
            if weight > 0:
                d['distance'] = 1.0 / weight
            else:
                d['distance'] = np.inf
                
        # Calculate node betweenness for this segment
        nb = nx.betweenness_centrality(G, weight='distance')
        
        for j, node in enumerate(node_labels):
            heatmap_data[i, j] = nb.get(node, 0)

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(15, 6)) # Since there are only 19 nodes, we don't need a super wide figure
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=node_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Node Betweenness'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Nodes)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(node_labels) + 1):
        # Extract the region initial (e.g., 'Fp1' -> 'F')
        r_prev = node_labels[i-1][0]
        r_curr = node_labels[i][0] if i < len(node_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(node_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot (y=1.02 in axes coordinates)
            ax.text(mid_x, 1.02, r_prev, transform=trans, ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Node Betweenness Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Nodes")
    plt.ylabel("Segments")

    # Rotate x-axis labels
    plt.xticks(rotation=90, fontsize=10)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_node_betweenness_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all node betweenness heatmaps with region labels!")


Saved: changePlots/node_betweenness/sub-NORB00064_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00065_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00069_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00070_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00075_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00077_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00078_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00079_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00080_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00081_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00082_ses-1_node_betweenness_heatmap.png
Saved: changePlots/node_betweenness/sub-NORB00083_ses-1_node_betw

## plot showing change in edge betweenness for each edge (pair of nodes)

In [ ]:
# Create a directory to save the edge betweenness plots
output_dir = Path("changePlots/edge_betweenness")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the edge betweenness for each segment
    # Shape: (12 segments, number of edges)
    heatmap_data = np.zeros((12, len(edge_labels)))

    # Populate the matrix with edge betweenness values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
        
        # Calculate distance: 1 / correlation
        for u, v, d in G.edges(data=True):
            weight = d.get('weight', 0)
            if weight > 0:
                d['distance'] = 1.0 / weight
            else:
                d['distance'] = np.inf
                
        # Calculate edge betweenness for this segment
        eb = nx.edge_betweenness_centrality(G, weight='distance')
        
        for j, edge in enumerate(edge_labels):
            u, v = edge.split('-')
            # Look up the edge betweenness regardless of node order (u,v vs v,u)
            heatmap_data[i, j] = eb.get((u, v), eb.get((v, u), 0))

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(30, 8))
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=edge_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Edge Betweenness'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Edges)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(edge_labels) + 1):
        # Extract the region initials (e.g., 'Fp1-F3' -> 'F-F')
        r_prev = f"{edge_labels[i-1].split('-')[0][0]}-{edge_labels[i-1].split('-')[1][0]}"
        r_curr = f"{edge_labels[i].split('-')[0][0]}-{edge_labels[i].split('-')[1][0]}" if i < len(edge_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(edge_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot (y=1.01 in axes coordinates)
            ax.text(mid_x, 1.01, r_prev, transform=trans, ha='center', va='bottom', fontsize=10, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Edge Betweenness Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Edges")
    plt.ylabel("Segments")

    # Rotate x-axis labels to fit all 171 edges legibly
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_edge_betweenness_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all edge betweenness heatmaps with region labels!")


Saved: changePlots/edge_betweenness/sub-NORB00064_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00065_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00069_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00070_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00075_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00077_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00078_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00079_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00080_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00081_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00082_ses-1_edge_betweenness_heatmap.png
Saved: changePlots/edge_betweenness/sub-NORB00083_ses-1_edge_betw

## plot showing change in node strength for each node

In [ ]:
# Create a directory to save the node strength plots
output_dir = Path("changePlots/node_strength")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

# Group nodes by region (F, C, T, P, O) to allow smooth cyan boundaries
node_labels = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'FZ', 
               'C3', 'C4', 'CZ', 'T3', 'T4', 'T5', 'T6', 
               'P3', 'P4', 'PZ', 'O1', 'O2']

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the node strength for each segment
    # Shape: (12 segments, 19 nodes)
    heatmap_data = np.zeros((12, len(node_labels)))

    # Populate the matrix with node strength values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
                
        # Calculate node strength for this segment
        ns = {node: G.degree(node, weight='weight') for node in G.nodes()}
        
        for j, node in enumerate(node_labels):
            heatmap_data[i, j] = ns.get(node, 0)

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(15, 6)) # 19 nodes don't need a super wide figure
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=node_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Node Strength'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Nodes)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(node_labels) + 1):
        # Extract the region initial (e.g., 'Fp1' -> 'F')
        r_prev = node_labels[i-1][0]
        r_curr = node_labels[i][0] if i < len(node_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(node_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot
            ax.text(mid_x, 1.02, r_prev, transform=trans, ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Node Strength Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Nodes")
    plt.ylabel("Segments")

    # Rotate x-axis labels
    plt.xticks(rotation=90, fontsize=10)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_node_strength_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all node strength heatmaps with region labels!")


Saved: changePlots/node_strength/sub-NORB00064_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00065_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00069_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00070_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00075_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00077_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00078_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00079_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00080_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00081_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00082_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00083_ses-1_node_strength_heatmap.png
Saved: changePlots/node_strength/sub-NORB00084_ses-1

## plot showing change in edge strength for each edge (pair of nodes)

In [ ]:
# Create a directory to save the edge strength plots
output_dir = Path("changePlots/edge_strength")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the edge strength for each segment
    # Shape: (12 segments, number of edges)
    heatmap_data = np.zeros((12, len(edge_labels)))

    # Populate the matrix with edge strength values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
        
        # Calculate nearest-neighbor edge centrality (edge strength)
        es = {}
        for u, v, d in G.edges(data=True):
            wk = d.get('weight', 0)
            cvs_u = G.degree(u, weight='weight')
            cvs_v = G.degree(v, weight='weight')
            es[(u, v)] = ((cvs_u + cvs_v - 2 * wk) / (abs(cvs_u - cvs_v) + 1)) * wk
            
        # Standardize edge tuple order to ensure consistency when looking it up
        normalized_es = {}
        for (u, v), val in es.items():
            edge_tuple = tuple(sorted([u, v]))
            normalized_es[edge_tuple] = val
        
        for j, edge in enumerate(edge_labels):
            u, v = edge.split('-')
            edge_tuple = tuple(sorted([u, v]))
            # Look up the edge strength
            heatmap_data[i, j] = normalized_es.get(edge_tuple, 0)

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(30, 8))
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=edge_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Edge Strength'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Edges)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(edge_labels) + 1):
        # Extract the region initials (e.g., 'Fp1-F3' -> 'F-F')
        r_prev = f"{edge_labels[i-1].split('-')[0][0]}-{edge_labels[i-1].split('-')[1][0]}"
        r_curr = f"{edge_labels[i].split('-')[0][0]}-{edge_labels[i].split('-')[1][0]}" if i < len(edge_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(edge_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot
            ax.text(mid_x, 1.01, r_prev, transform=trans, ha='center', va='bottom', fontsize=10, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Edge Strength Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Edges")
    plt.ylabel("Segments")

    # Rotate x-axis labels legibly
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_edge_strength_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all edge strength heatmaps with region labels!")


Saved: changePlots/edge_strength/sub-NORB00064_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00065_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00069_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00070_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00075_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00077_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00078_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00079_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00080_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00081_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00082_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00083_ses-1_edge_strength_heatmap.png
Saved: changePlots/edge_strength/sub-NORB00084_ses-1

## plot showing change in node eigenvector for each node

In [ ]:
# Create a directory to save the node eigenvector centrality plots
output_dir = Path("changePlots/node_eigenvector")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

# Group nodes by region (F, C, T, P, O) to allow smooth cyan boundaries
node_labels = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'FZ', 
               'C3', 'C4', 'CZ', 'T3', 'T4', 'T5', 'T6', 
               'P3', 'P4', 'PZ', 'O1', 'O2']

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the node eigenvector centrality for each segment
    # Shape: (12 segments, 19 nodes)
    heatmap_data = np.zeros((12, len(node_labels)))

    # Populate the matrix with node eigenvector values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
                
        # Calculate eigenvector centrality for this segment
        try:
            ec = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            # Fallback to the exact numpy solver if power iteration fails to converge
            ec = nx.eigenvector_centrality_numpy(G, weight='weight')
        
        for j, node in enumerate(node_labels):
            heatmap_data[i, j] = ec.get(node, 0)

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(15, 6)) # 19 nodes don't need a super wide figure
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=node_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Node Eigenvector Centrality'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Nodes)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(node_labels) + 1):
        # Extract the region initial (e.g., 'Fp1' -> 'F')
        r_prev = node_labels[i-1][0]
        r_curr = node_labels[i][0] if i < len(node_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(node_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot
            ax.text(mid_x, 1.02, r_prev, transform=trans, ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Node Eigenvector Centrality Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Nodes")
    plt.ylabel("Segments")

    # Rotate x-axis labels
    plt.xticks(rotation=90, fontsize=10)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_node_eigenvector_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all node eigenvector heatmaps with region labels!")


Saved: changePlots/node_eigenvector/sub-NORB00064_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00065_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00069_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00070_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00075_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00077_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00078_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00079_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00080_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00081_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00082_ses-1_node_eigenvector_heatmap.png
Saved: changePlots/node_eigenvector/sub-NORB00083_ses-1_node_eige

## plot showing change in edge eigenvector for each edge (pair of nodes)

In [ ]:
# Function to calculate edge eigenvector centrality
def compute_edge_eigenvector_centrality(G, weight='weight'):
    L = nx.line_graph(G)
    for e in L.edges():
        edge1, edge2 = e
        w1 = G[edge1[0]][edge1[1]].get(weight, 0)
        w2 = G[edge2[0]][edge2[1]].get(weight, 0)
        L[edge1][edge2]['weight'] = (w1 + w2) / 2.0
    try:
        ec = nx.eigenvector_centrality(L, weight='weight', max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        ec = nx.eigenvector_centrality_numpy(L, weight='weight')
    
    normalized_ec = {}
    for edge_tuple, val in ec.items():
        normalized_ec[tuple(sorted(list(edge_tuple)))] = val
    return normalized_ec

# Create a directory to save the edge eigenvector centrality plots
output_dir = Path("changePlots/edge_eigenvector")
output_dir.mkdir(parents=True, exist_ok=True)

# The y-axis labels remain the same for all plots
yticklabels = [f"Segment {i}" for i in range(12, 0, -1)]

for subj in formatted_subjects:
    subj_dir = abs_segment_base / subj / "ses-1"
    
    # Check that directory exists
    if not subj_dir.exists():
        print(f"Skipping {subj}: Directory not found.")
        continue
        
    csv_files = sorted(subj_dir.glob('*.csv'))
    
    # Ensure there are at least 12 segments
    if len(csv_files) < 12:
        print(f"Skipping {subj}: Found {len(csv_files)} segments (requires 12).")
        continue
        
    # Take the first 12 segments
    csv_files = csv_files[:12]
    
    # Initialize a matrix to hold the edge eigenvector centrality for each segment
    # Shape: (12 segments, number of edges)
    heatmap_data = np.zeros((12, len(edge_labels)))

    # Populate the matrix with edge eigenvector values
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file, index_col=0)
        G = nx.from_pandas_adjacency(df)
                
        # Calculate edge eigenvector centrality for this segment
        ec = compute_edge_eigenvector_centrality(G, weight='weight')
        
        for j, edge in enumerate(edge_labels):
            u, v = edge.split('-')
            edge_tuple = tuple(sorted([u, v]))
            # Look up the edge eigenvector centrality
            heatmap_data[i, j] = ec.get(edge_tuple, 0)

    # Segment 1 should be at the bottom, so we reverse the row order
    heatmap_data = heatmap_data[::-1, :]

    # Plot the heatmap
    plt.figure(figsize=(30, 8))
    ax = sns.heatmap(
        heatmap_data, 
        cmap='hot', 
        xticklabels=edge_labels, 
        yticklabels=yticklabels, 
        cbar_kws={'label': 'Edge Eigenvector Centrality'}
    )

    # ---------------------------------------------------------
    # Region Grouping & Labeling Logic (Edges)
    # ---------------------------------------------------------
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)

    start_idx = 0
    for i in range(1, len(edge_labels) + 1):
        # Extract the region initials (e.g., 'Fp1-F3' -> 'F-F')
        r_prev = f"{edge_labels[i-1].split('-')[0][0]}-{edge_labels[i-1].split('-')[1][0]}"
        r_curr = f"{edge_labels[i].split('-')[0][0]}-{edge_labels[i].split('-')[1][0]}" if i < len(edge_labels) else None
        
        if r_prev != r_curr:
            # 1. Draw vertical boundary line between region groups
            if i < len(edge_labels):
                ax.axvline(x=i, color='cyan', linestyle='--', linewidth=1.5)
                
            # 2. Calculate horizontal center of this region group
            mid_x = (start_idx + i) / 2.0
            
            # 3. Place label slightly above the top edge of the plot
            ax.text(mid_x, 1.01, r_prev, transform=trans, ha='center', va='bottom', fontsize=10, fontweight='bold')
            
            start_idx = i
    # ---------------------------------------------------------

    plt.title(f"Segment-wise Edge Eigenvector Centrality Change for {subj} (Session 1)", pad=35)
    plt.xlabel("Edges")
    plt.ylabel("Segments")

    # Rotate x-axis labels legibly
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0)

    plt.tight_layout()
    
    # Save the plot
    save_path = output_dir / f"{subj}_ses-1_edge_eigenvector_heatmap.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Close the figure to free up memory
    plt.close()

print("Finished saving all edge eigenvector heatmaps with region labels!")


Saved: changePlots/edge_eigenvector/sub-NORB00064_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00065_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00069_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00070_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00075_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00077_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00078_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00079_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00080_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00081_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00082_ses-1_edge_eigenvector_heatmap.png
Saved: changePlots/edge_eigenvector/sub-NORB00083_ses-1_edge_eige